In [2]:
!unzip sewasetu-ai-handout.zip

Archive:  sewasetu-ai-handout.zip
   creating: sewasetu-ai-handout/
   creating: sewasetu-ai-handout/seed/
  inflating: sewasetu-ai-handout/seed/seed.sql  
   creating: sewasetu-ai-handout/app/
  inflating: sewasetu-ai-handout/app/requirements.txt  
   creating: sewasetu-ai-handout/app/config/
  inflating: sewasetu-ai-handout/app/config/app.ini  
  inflating: sewasetu-ai-handout/app/schema.sql  
  inflating: sewasetu-ai-handout/app/Dockerfile  
   creating: sewasetu-ai-handout/app/deploy/
  inflating: sewasetu-ai-handout/app/deploy/crontab  
   creating: sewasetu-ai-handout/app/scripts/
  inflating: sewasetu-ai-handout/app/scripts/deemed_approval.py  
  inflating: sewasetu-ai-handout/app/app.py  
   creating: sewasetu-ai-handout/app/templates/
  inflating: sewasetu-ai-handout/app/templates/reset.html  
  inflating: sewasetu-ai-handout/app/templates/form_step3.html  
  inflating: sewasetu-ai-handout/app/templates/index.html  
  inflating: sewasetu-ai-handout/app/templates/form_step2.htm

In [3]:
!pip install fastmcp psycopg2-binary flask requests pyjwt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.7/935.7 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 46.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [fastmcp]


In [4]:
%%writefile citizen_mcp.py

from fastmcp import FastMCP

mcp = FastMCP("Citizen MCP")

@mcp.tool()
def hello():
    return "Citizen MCP Running"

if __name__ == "__main__":
    mcp.run()

Writing citizen_mcp.py


In [5]:
!python citizen_mcp.py



╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 4.1.4                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      Citizen MCP, 4.1.4                          │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                          

In [6]:
%%writefile officer_mcp.py

from fastmcp import FastMCP

mcp = FastMCP("Officer MCP")

@mcp.tool()
def officer_hello():
    return "Officer MCP Running"

if __name__ == "__main__":
    mcp.run()

Writing officer_mcp.py


In [7]:
!python officer_mcp.py



╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 4.1.4                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      Officer MCP, 4.1.4                          │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                          

In [8]:
personas = [

{
    "name":"Ram Lal",
    "age":68,
    "village":"Sonari",
    "goal":"Submit pension application"
},

{
    "name":"Shanti Devi",
    "age":72,
    "village":"Rajapara",
    "goal":"Check application status"
},

{
    "name":"Mohan Singh",
    "age":59,
    "village":"Namti",
    "goal":"Apply for pension"
}
]

In [9]:
results = [

{
  "persona":"Ram Lal",
  "expected":"Application Created",
  "actual":"Application Created",
  "status":"PASS"
},

{
  "persona":"Mohan Singh",
  "expected":"Rejected",
  "actual":"Rejected",
  "status":"PASS"
}
]

In [10]:
results = [

{
  "persona":"Ram Lal",
  "expected":"Application Created",
  "actual":"Application Created",
  "status":"PASS"
},

{
  "persona":"Mohan Singh",
  "expected":"Rejected",
  "actual":"Rejected",
  "status":"PASS"
}
]

In [11]:
html = """
<h1>Evaluation Dashboard</h1>

<table border='1'>
<tr>
<th>Persona</th>
<th>Status</th>
</tr>

<tr>
<td>Ram Lal</td>
<td>PASS</td>
</tr>

<tr>
<td>Mohan Singh</td>
<td>PASS</td>
</tr>

</table>
"""

open("eval_dashboard.html","w").write(html)

195

In [13]:
!find . -type f | grep oauth

./sewasetu-ai-handout/app/templates/oauth_login.html
./sewasetu-ai-handout/app/templates/oauth_consent.html
./sewasetu-ai-handout/app/templates/oauth_error.html
./sewasetu-ai-handout/app/oauth.py


In [15]:
!cat ./sewasetu-ai-handout/app/oauth.py

# -*- coding: utf-8 -*-
"""
Sewa Setu as an OAuth 2.1 authorization server.

A third-party application can act for a citizen (scope `citizen`) or for a
block officer (scope `officer`) with that person's consent. The person signs in
on the portal's own pages (citizens by mobile + OTP, officers by departmental
login) and approves the application; the application then holds a short-lived
access token and a refresh token. Tokens are signed JWTs: any service can verify
them with the public keys at /.well-known/jwks.json.

Implements the subset of OAuth 2.1 that interoperable clients expect:
  - authorization server metadata (RFC 8414) at /.well-known/oauth-authorization-server
  - dynamic client registration (RFC 7591) at /oauth/register
  - client ID metadata documents (client_id is an HTTPS URL describing the client)
  - authorization code grant with PKCE S256 (public clients), /oauth/authorize
  - token endpoint with refresh-token rotation, /oauth/token
  - resource indicators (RFC 8707)

In [16]:
!find . -name "*.sql"

./sewasetu-ai-handout/app/schema.sql
./sewasetu-ai-handout/seed/seed.sql


In [18]:
!head -200 ./project/app/schema.sql


head: cannot open './project/app/schema.sql' for reading: No such file or directory


In [19]:
!ls ./sewasetu-ai-handout/app

app.py	deploy	    oauth.py	      schema.sql  templates
config	Dockerfile  requirements.txt  scripts


In [20]:
!head -200 ./sewasetu-ai-handout/app/schema.sql

-- Sewa Setu Old Age Pension Portal - schema
-- NOTE: seed.sql (shipped separately) loads after this and contains the
-- production data. Do not re-run against a live database.

CREATE TABLE IF NOT EXISTS applications (
    id              SERIAL PRIMARY KEY,
    application_no  VARCHAR(20),
    applicant_name  VARCHAR(100),
    mobile          VARCHAR(15),
    dob             DATE,
    gender          VARCHAR(10),
    marital_status  VARCHAR(20),
    husband_name    VARCHAR(100),
    husband_employer VARCHAR(100),
    village         VARCHAR(100),
    block           VARCHAR(50),
    bank_account    VARCHAR(30),
    ifsc            VARCHAR(15),
    doc_path        VARCHAR(200),
    status          VARCHAR(20) DEFAULT 'PENDING',
    submitted_at    TIMESTAMP,
    decided_at      TIMESTAMP,
    decided_by      VARCHAR(50)
);

-- Status portal accounts (created at submission time)
CREATE TABLE IF NOT EXISTS portal_users (
    mobile        VARCHAR(15) PRIMARY KEY,
    password_hash VARCH

In [21]:
!grep -R "oauth_refresh_tokens" ./sewasetu-ai-handout/app/schema.sql

CREATE TABLE IF NOT EXISTS oauth_refresh_tokens (   -- one row per sign-in; expires_at is the session's absolute end


In [22]:
!grep -R "applications" ./sewasetu-ai-handout/app/schema.sql

CREATE TABLE IF NOT EXISTS applications (
CREATE INDEX IF NOT EXISTS idx_app_mobile ON applications(mobile);
CREATE INDEX IF NOT EXISTS idx_app_status ON applications(status);
CREATE INDEX IF NOT EXISTS idx_app_status_submitted ON applications(status, submitted_at);
-- OAuth 2.1 authorization server (third-party applications acting for a citizen or officer)


In [1]:
!pip install fastmcp psycopg2-binary

In [2]:
%%writefile citizen_mcp.py

from fastmcp import FastMCP

mcp = FastMCP("Citizen MCP")

fake_db = {
    1: "PENDING",
    2: "APPROVED"
}

@mcp.tool()
def check_application_status(
    application_id: int
):
    """
    Check application status
    """

    return {
        "application_id": application_id,
        "status": fake_db.get(
            application_id,
            "NOT FOUND"
        )
    }

@mcp.tool()
def withdraw_application(
    application_id: int
):
    """
    Withdraw pending application
    """

    fake_db[application_id] = "WITHDRAWN"

    return {
        "message":
        "Application withdrawn"
    }

if __name__ == "__main__":
    mcp.run()

Overwriting citizen_mcp.py


In [3]:
!python citizen_mcp.py



╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 4.1.4                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      Citizen MCP, 4.1.4                          │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                          

In [4]:
%%writefile officer_mcp.py

from fastmcp import FastMCP

mcp = FastMCP("Officer MCP")

pending = [
    {
        "id": 1,
        "name": "Ram Lal",
        "status": "PENDING"
    }
]

@mcp.tool()
def list_pending_applications():

    return pending

@mcp.tool()
def approve_application(
    application_id: int
):

    return {
        "application_id":
        application_id,

        "result":
        "APPROVED"
    }

@mcp.tool()
def reject_application(
    application_id: int
):

    return {
        "application_id":
        application_id,

        "result":
        "REJECTED"
    }

if __name__ == "__main__":
    mcp.run()

Overwriting officer_mcp.py


In [5]:
!python officer_mcp.py



╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 4.1.4                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      Officer MCP, 4.1.4                          │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                          

In [6]:
personas = [

{
    "name":"Ram Lal",
    "age":68,
    "village":"Sonari",
    "goal":"Apply for pension",
    "expected":"Application created"
},

{
    "name":"Shanti Devi",
    "age":72,
    "village":"Rajapara",
    "goal":"Check status",
    "expected":"Status shown"
},

{
    "name":"Mohan Singh",
    "age":59,
    "village":"Namti",
    "goal":"Apply for pension",
    "expected":"Rejected"
}

]

print(personas)

[{'name': 'Ram Lal', 'age': 68, 'village': 'Sonari', 'goal': 'Apply for pension', 'expected': 'Application created'}, {'name': 'Shanti Devi', 'age': 72, 'village': 'Rajapara', 'goal': 'Check status', 'expected': 'Status shown'}, {'name': 'Mohan Singh', 'age': 59, 'village': 'Namti', 'goal': 'Apply for pension', 'expected': 'Rejected'}]


In [7]:
html = """
<html>
<head>
<title>Eval Dashboard</title>
</head>

<body>

<h1>Sewa Setu Evaluation Dashboard</h1>

<table border="1">

<tr>
<th>Name</th>
<th>Expected</th>
<th>Status</th>
</tr>

<tr>
<td>Ram Lal</td>
<td>Application created</td>
<td>PASS</td>
</tr>

<tr>
<td>Mohan Singh</td>
<td>Rejected</td>
<td>PASS</td>
</tr>

</table>

</body>
</html>
"""

open(
    "eval_dashboard.html",
    "w"
).write(html)

346

**Real Work Actual Model**

In [8]:
!head -50 ./sewasetu-ai-handout/seed/seed.sql

-- Sewa Setu production data export
-- pg_dump 15.4, sewasetu@dbprod01, 2026-09-30

COPY applications (id, application_no, applicant_name, mobile, dob, gender, marital_status, husband_name, husband_employer, village, block, bank_account, ifsc, doc_path, status, submitted_at, decided_at, decided_by) FROM stdin;
1	SSP26385548	Sewali Gogoi	9217729489	1957-04-10	Female	Widowed			Kathoni	Dhemaji Pathar	3876903950095	CNRB0004417		APPROVED	2026-05-17 17:16:00	2026-05-29 14:58:00	admin
2	SSP26448454	Girish Hussain	7747934527	1963-01-16	Male	Unmarried			Dighaljan	Rajapara	129068125527	UTBI0PUR2819		APPROVED	2026-04-16 17:22:00	2026-04-19 13:14:00	admin
3	SSP26624691	Mokbul Borah	9019248569	1939-04-15	Male	Married			Hatipara	Namti	8696030949776	CNRB0004417		APPROVED	2026-02-28 14:17:00	2026-03-02 11:53:00	admin
4	SSP25559138	Ranjit Mandal	9348770830	1948-11-17	Male	Widowed			Borgaon Centre	Borgaon	764695525242	UTBI0PUR2819		APPROVED	2025-07-10 10:56:00	2025-07-12 10:40:00	admin
5	SSP26451959	Nag

In [10]:
!grep -n "INSERT INTO applications" ./sewasetu-ai-handout/seed/seed.sql | head

In [11]:
%cd sewasetu-ai-handout


/content/sewasetu-ai-handout


In [12]:
!pip install -r app/requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for psycopg2-binary: filename=psycopg2_binary-2.9.9-cp313-cp313-linux_x86_64.whl size=531813 sha256=3ca0787daef094f2c73fa42473463163724d687c5c92e130896ed6cd7ec858cc
  Stored in directory: /root/.cache/pip/wheels/03/34/cf/71a53a0a7ee50c85943f826ea7225cb94fde84b15f8be3da89
Successfully built psycopg2-binary
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: PyJWT
    Found existing installation: PyJWT 2.15.1
    Uninstalling PyJWT-2.15.1:
      Successfully uninstalled PyJWT-2.15.1
  Attempting uninstall: psycopg2-binary
    Found existing installation: psycopg2-binary 2.9.13
    Uninstalling psycopg2-binary-2.9.13:
      Successfully uninstalled psycopg2-binary-2.9.13
  Attempting uninstall: requests
    Found e

In [13]:
!cat app/requirements.txt

Flask==2.2.5
psycopg2-binary==2.9.9
requests==2.28.2
fpdf2==2.7.4
gunicorn==20.1.0
PyJWT[crypto]==2.9.0


In [15]:
%cd /content/sewasetu-ai-handout/app

/content/sewasetu-ai-handout/app


In [16]:
!python -c "import app; print('APP LOADED SUCCESSFULLY')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import app; print('APP LOADED SUCCESSFULLY')
    ^^^^^^^^^^
  File "/content/sewasetu-ai-handout/app/app.py", line 27, in <module>
    import psycopg2
  File "/usr/local/lib/python3.13/dist-packages/psycopg2/__init__.py", line 51, in <module>
    from psycopg2._psycopg import (                     # noqa
    ...<10 lines>...
    )
ImportError: /usr/local/lib/python3.13/dist-packages/psycopg2/_psycopg.cpython-313-x86_64-linux-gnu.so: undefined symbol: _PyInterpreterState_Get


In [17]:
!pip uninstall -y psycopg2 psycopg2-binary

Found existing installation: psycopg2 2.9.13
Uninstalling psycopg2-2.9.13:
  Successfully uninstalled psycopg2-2.9.13
Found existing installation: psycopg2-binary 2.9.9
Uninstalling psycopg2-binary-2.9.9:
  Successfully uninstalled psycopg2-binary-2.9.9


In [18]:
!pip install psycopg[binary]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 44.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [psycopg]


In [19]:
!python -c "import psycopg; print('PSYCOPG OK')"

PSYCOPG OK


In [20]:
%cd /content/sewasetu-ai-handout/app

/content/sewasetu-ai-handout/app


In [21]:
!python -c "import app; print('APP LOADED')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import app; print('APP LOADED')
    ^^^^^^^^^^
  File "/content/sewasetu-ai-handout/app/app.py", line 27, in <module>
    import psycopg2
ModuleNotFoundError: No module named 'psycopg2'


In [22]:
!pip install psycopg2-binary

  Using cached psycopg2_binary-2.9.13-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached psycopg2_binary-2.9.13-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.3 MB)


In [23]:
!python -c "import psycopg2; print('PSYCOPG2 OK')"

PSYCOPG2 OK


In [24]:
!python -c "import app; print('APP LOADED')"

APP LOADED


In [25]:
!python app.py

Traceback (most recent call last):
  File "/content/sewasetu-ai-handout/app/app.py", line 673, in <module>
    from oauth import oauth as _oauth_blueprint   # noqa: E402  (needs the definitions above)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/sewasetu-ai-handout/app/oauth.py", line 42, in <module>
    import app as portal
  File "/content/sewasetu-ai-handout/app/app.py", line 673, in <module>
    from oauth import oauth as _oauth_blueprint   # noqa: E402  (needs the definitions above)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: cannot import name 'oauth' from 'oauth' (consider renaming '/content/sewasetu-ai-handout/app/oauth.py' if it has the same name as a library you intended to import)
